# 01 — Hugging Face Transformers Baseline

Goal: measure a naive sequential baseline using `transformers.generate()`. This is intentionally not a production serving pattern. The point is to create a reference before using vLLM.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from vllm_lab.benchmark import benchmark_transformers, summarize_results
from vllm_lab.utils import load_config, get_gpu_snapshot

config = load_config(repo_root / 'configs' / 'lab_config.yaml')
model_name = config['model']['default_name']
prompts = config['benchmark']['prompts']
max_new_tokens = config['model']['max_new_tokens']

print('model:', model_name)
print('gpu:', get_gpu_snapshot())

Load tokenizer and model. For a larger model, this is where you will first hit VRAM limits. If this cell fails, do not blame vLLM yet; first verify the model fits at all.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

Run sequential generation. This processes one prompt after another. It is easy to write, but it usually underuses the GPU for serving workloads.

In [ ]:
results = benchmark_transformers(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    max_new_tokens=max_new_tokens,
    device=device,
)

df = pd.DataFrame([r.to_dict() for r in results])
df

In [ ]:
summary = summarize_results(results)
summary

In [ ]:
out = repo_root / 'results' / 'hf_transformers_baseline.csv'
out.parent.mkdir(exist_ok=True)
df.to_csv(out, index=False)
print('wrote', out)

What to notice: the baseline is simple, but not memory-aware, not request-scheduler-aware, and not designed for continuous batching. It is a useful baseline, not a serving solution.